In [3]:
from grakel.datasets import fetch_dataset
MUTAG = fetch_dataset("MUTAG", verbose=False)
G = MUTAG.data
y = MUTAG.target


In [2]:
from grakel import GraphKernel
from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

# Define kernel
sp_kernel = GraphKernel(kernel="edge_histogram", normalize=True)

# 10-fold splitter
kf = KFold(n_splits=10, shuffle=True, random_state=42)

accuracies = []

for train_index, test_index in kf.split(G):
    
    # Split graphs and labels
    G_train = [G[i] for i in train_index]
    G_test  = [G[i] for i in test_index]
    
    y_train = y[train_index]
    y_test  = y[test_index]
    
    # Compute kernel matrices
    K_train = sp_kernel.fit_transform(G_train)     # (n_train x n_train)
    K_test  = sp_kernel.transform(G_test)          # (n_test x n_train)
    
    # Train SVM
    clf = SVC(kernel='precomputed')
    clf.fit(K_train, y_train)
    
    # Predict
    y_pred = clf.predict(K_test)
    
    # Store accuracy
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

print("Fold accuracies:", np.round(accuracies, 4))
print("Mean accuracy: %.4f" % np.mean(accuracies))
print("Std accuracy:  %.4f" % np.std(accuracies))



Fold accuracies: [0.6842 0.7368 0.7368 0.6316 0.6842 0.6316 0.6842 0.7368 0.6111 0.8333]
Mean accuracy: 0.6971
Std accuracy:  0.0631
